# Impedance Matching

## 1 Maximum Power Transfer

In high-frequency electronics, the goal is often to deliver maximum power from a source to a load. For a source with impedance $Z_s = R_s + jX_s$, maximum power transfer occurs when the load impedance is the complex conjugate:

$$
Z_L = Z_s^* = R_s - jX_s
$$

If we consider the resistive part of the load, the **normalized power delivered to the load** can be written as

$$
P_L = \frac{R_L}{(R_s + R_L)^2}
$$

This expression shows that the delivered power depends on the ratio between the source resistance and the load resistance. Maximum power transfer occurs when:

$$
R_L = R_s
$$

which corresponds to the conjugate matching condition in the general complex case.In AC circuits, perfect matching occurs at only one frequency.
)


## 2 The L-Network

The **L-network** is the simplest matching circuit, consisting of two reactive elements.  
There are **eight possible configurations**:

- 4 Low-pass
- 4 High-pass

The shunt element is always placed across the larger of the two resistances ($R_s$ or $R_L$).

**Quality Factor ($Q$)**

For a simple $L$-network, the node $Q$ is defined by:

$$
Q = \sqrt{\frac{R_{high}}{R_{low}} - 1}
$$

---

## 1.3 Complex Matching

When the source or load contains reactive components ($L$ or $C$), we use two primary methods:

### Absorption
The stray reactances are **absorbed into the calculated values** of the matching network.

### Resonance
An **equal and opposite reactance** is added to the network to cancel out the stray reactance, effectively reducing the problem to a **real-to-real impedance match**.

# Maximum Power Transfer

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def plot_power_transfer_linear(rs_fixed):
    # Create a linear range of Load Resistances centered around Rs
    # We go from 0 to 4 times the source resistance to see the drop-off
    rl = np.linspace(0.1, rs_fixed * 4, 1000)
    
    # Normalized power calculation: P = Rl / (Rs + Rl)^2
    # Normalized such that the peak at Rs = Rl is 1.0
    p_max = 1 / (4 * rs_fixed)
    pl = rl / (rs_fixed + rl)**2
    pl_norm = pl / p_max
    
    plt.figure(figsize=(10, 5))
    
    # Linear plot
    plt.plot(rl, pl_norm, color='blue', linewidth=2, label='Delivered Power')
    
    # Highlight the matching point
    plt.axvline(rs_fixed, color='red', linestyle='--', alpha=0.7)
    plt.plot(rs_fixed, 1.0, 'ro') # Dot at the peak
    
    # Formatting with raw strings for LaTeX
    plt.title(r'Power Transfer Efficiency (Linear Scale, $R_s$ = ' + str(rs_fixed) + r' $\Omega$)', fontsize=14)
    plt.xlabel(r'Load Resistance $R_L$ ($\Omega$)', fontsize=12)
    plt.ylabel(r'Normalized Power ($P_L / P_{max}$)', fontsize=12)
    plt.grid(True, ls="-", alpha=0.3)
    plt.xlim(0, rs_fixed * 4)
    plt.ylim(0, 1.1)
    
    # Annotation
    plt.annotate(r'Maximum Power ($R_L = R_s$)', 
                 xy=(rs_fixed, 1.0), 
                 xytext=(rs_fixed * 1.5, 0.8),
                 arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    
    plt.legend()
    plt.show()

# Setup UI
rs_slider = widgets.IntSlider(
    value=50, 
    min=10, 
    max=200, 
    step=10, 
    description=r'$R_s$ ($\Omega$):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

out = widgets.interactive_output(plot_power_transfer_linear, {'rs_fixed': rs_slider})

display(rs_slider, out)

IntSlider(value=50, description='$R_s$ ($\\Omega$):', layout=Layout(width='50%'), max=200, min=10, step=10, st…

Output()

# L Impedance Matching (5., 6 eta 7 Ariketa)

In [27]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import schemdraw
import schemdraw.elements as elm

def calculateLNetwork(rsS, xsS, rlL, xlL, freqMhz, filterType='Low-Pass'):
    w = 2 * np.pi * freqMhz * 1e6
    
    # 1. Real-to-real matching logic
    if rsS < rlL:
        topology = "Shunt-Load"
        q = np.sqrt((rlL / rsS) - 1)
        xsReq = q * rsS
        xpReq = rlL / q
    else:
        topology = "Shunt-Source"
        q = np.sqrt((rsS / rlL) - 1)
        xsReq = q * rlL
        xpReq = rsS / q

    # 2. Assign base reactances based on filter type
    if filterType == 'Low-Pass':
        xsBase, xpBase = xsReq, -xpReq 
    else:
        xsBase, xpBase = -xsReq, xpReq 
        
    # 3. Absorption Logic
    # Series arm: xsBase is the required matching reactance. 
    # To match a source with xsS, we need xsFinal = xsBase - xsS
    xsFinal = xsBase - xsS
    
    if topology == "Shunt-Load" and xlL != 0:
        bMatch = 1 / xpBase
        bLoad = 1 / xlL
        bFinal = bMatch - bLoad
        xpFinal = 1 / bFinal if bFinal != 0 else 1e12 
    else:
        xpFinal = xpBase

    def getComp(x, w):
        if abs(x) < 1e-9: return ('L', 0) # Neutral
        if x > 0: return ('L', x / w)
        else: return ('C', -1 / (w * x))

    sType, sVal = getComp(xsFinal, w)
    pType, pVal = getComp(xpFinal, w)
    
    return q, topology, sType, sVal, pType, pVal

def updateMatchingUI(rsS, xsS, rlL, xlL, freqMhz, filterType):
    q, topo, sType, sVal, pType, pVal = calculateLNetwork(rsS, xsS, rlL, xlL, freqMhz, filterType)
    
    display(Markdown(f"### Matching Results ({filterType})"))
    display(Markdown(f"- **Calculated Q:** {q:.2f} | **Topology:** {topo}"))
    
    d = schemdraw.Drawing()
    
    # --- Source Section ---
    Vsig = d.add(elm.SourceSin().up().label('Vs').at((0,0)))
    gndY = Vsig.start[1] 
    
    d.add(elm.Resistor().right().label(f'{rsS:.2f}Ω'))
    
    # Handle signed Series Source Reactance (Inductor if +, Capacitor if -)
    if xsS != 0:
        elXs = elm.Inductor() if xsS > 0 else elm.Capacitor()
        d.add(elXs.right().label(f'j{xsS:.2f}Ω'))

    # --- Matching Network ---
    if topo == "Shunt-Source":
        topNode = d.here
        elP = elm.Capacitor() if pType == 'C' else elm.Inductor()
        labelP = f'{pVal*1e12:.2f}pF' if pType == 'C' else f'{pVal*1e9:.2f}nH'
        pComp = d.add(elP.down().label(labelP))
        d.add(elm.Line().at(pComp.end).toy(Vsig.start))
        
        d.move_from(topNode)
        elS = elm.Inductor() if sType == 'L' else elm.Capacitor()
        labelS = f'{sVal*1e9:.2f}nH' if sType == 'L' else f'{sVal*1e12:.2f}pF'
        d.add(elS.right().label(labelS))
    else:
        elS = elm.Inductor() if sType == 'L' else elm.Capacitor()
        labelS = f'{sVal*1e9:.2f}nH' if sType == 'L' else f'{sVal*1e12:.2f}pF'
        d.add(elS.right().label(labelS))
        
        topNode = d.here
        elP = elm.Capacitor() if pType == 'C' else elm.Inductor()
        labelP = f'{pVal*1e12:.2f}pF' if pType == 'C' else f'{pVal*1e9:.2f}nH'
        pComp = d.add(elP.down().label(labelP))
        d.add(elm.Line().at(pComp.end).toy(Vsig.start))
        d.move_from(topNode)

    # --- Parallel Load Section ---
    d.add(elm.Line().right().length(1.5)) 
    topLoadR = d.here
    L1 = d.add(elm.Resistor().down().label(f'{rlL:.2f}Ω'))
    d.add(elm.Line().at(L1.end).toy(Vsig.start))
    
    if xlL != 0:
        d.add(elm.Line().right().at(topLoadR).length(3.0)) 
        elL = elm.Inductor() if xlL > 0 else elm.Capacitor()
        L2 = d.add(elL.down().label(f'j{xlL:.2f}Ω'))
        d.add(elm.Line().at(L2.end).toy(Vsig.start))
        groundEndX = L2.end[0]
    else:
        groundEndX = L1.end[0]

    # Final Ground Rail
    d.add(elm.Line().at((groundEndX, gndY)).tox(Vsig.start))
    
    display(d.draw())

# UI Components (No underscores)
rsSlider = widgets.FloatSlider(value=50, min=5, max=500, description='Rs (Ω)')
xsSlider = widgets.FloatSlider(value=0, min=-500, max=500, description='Xs Ser (Ω)')
rlSlider = widgets.FloatSlider(value=200, min=5, max=1000, description='Rl (Ω)')
xlSlider = widgets.FloatSlider(value=0, min=-5000, max=5000, description='Xl Par (Ω)')
freqSlider = widgets.FloatSlider(value=100, min=1, max=2000, description='Freq (MHz)')
typeDropdown = widgets.Dropdown(options=['Low-Pass', 'High-Pass'], value='Low-Pass', description='Type')

uiBox = widgets.VBox([
    widgets.HBox([rsSlider, xsSlider]),
    widgets.HBox([rlSlider, xlSlider]),
    widgets.HBox([freqSlider, typeDropdown])
])

outArea = widgets.interactive_output(updateMatchingUI, {
    'rsS': rsSlider, 'xsS': xsSlider, 'rlL': rlSlider, 'xlL': xlSlider, 'freqMhz': freqSlider, 'filterType': typeDropdown
})

display(uiBox, outArea)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<IPython.core.display.Markdown object>…

# Smith Chart

## Impedance and Admitance Placement

In [48]:
import numpy as np
import matplotlib.pyplot as plt
import skrf as rf
import ipywidgets as widgets
from IPython.display import display

def showSmithBasics(r_val, x_val, show_z=True, show_y=True):
    # 1. Reinicio total del lienzo
    plt.close('all')
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # 2. Dibujar Grilla Z (Roja en tu código)
    if show_z:
        rf.plotting.smith(ax=ax, chart_type='z', draw_labels=True)
        # Cambiar color de todas las líneas
        for line in ax.lines:
            line.set_color('red')
            line.set_alpha(0.35)

        # Cambiar color de círculos (patches)
        for patch in ax.patches:
            patch.set_edgecolor('red')
            patch.set_alpha(0.35)
    
    # Guardamos cuántas líneas hay de Z
    num_z_lines = len(ax.get_lines())
    num_z_patches = len(ax.patches)

    # 3. Dibujar Grilla Y (Azul en tu código)
    if show_y:
        rf.plotting.smith(ax=ax, chart_type='y', draw_labels=False)

        for line in ax.lines[num_z_lines:]:
            line.set_color('blue')
            line.set_alpha(0.35)

        for patch in ax.patches[num_z_patches:]:
            patch.set_edgecolor('blue')
            patch.set_alpha(0.35)

    # 4. Limpiar líneas "fantasma" (ejes que cruzan el diámetro)
    for line in ax.get_lines():
        if line.get_color() in ['k', 'black', 'grey', '#000000']:
            line.set_alpha(0.1) 

    # 5. Punto seleccionado y Cálculos de Admitancia
    z0 = 50
    z_point = complex(r_val, x_val)
    z_norm = z_point / z0
    gamma = (z_norm - 1) / (z_norm + 1)
    
    # Formateo de Z con dos decimales
    z_label = f"Z = {r_val:.2f} + j{x_val:.2f} Ohms"
    
    # Calculamos Y si Z no es cero para evitar división por cero
    if z_point != 0:
        y_point = 1 / z_point
        # Formateo de Y con dos decimales en mili-Siemens
        y_label = f"Y = {y_point.real*1000:.2f} + j{y_point.imag*1000:.2f} mS"
    else:
        y_label = "Y = inf (Short Circuit)"
    
    # Dibujar el punto con Z e Y en la leyenda (limitado a 2 decimales)
    ax.plot([gamma.real], [gamma.imag], 'ko', markersize=12, zorder=100,
            label=f'{z_label}\n{y_label}')

    # Estética
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect('equal')
    
    # Forzar color de las etiquetas de los círculos
    for text in ax.texts:
        text.set_color('blue')
        text.set_weight('bold')

    ax.set_title("Smith Chart (Z e Y)", fontsize=16, pad=20)
    ax.legend(loc='upper right', frameon=True, shadow=True, fontsize=10)
    
    plt.show()

# Controles
r_s = widgets.FloatSlider(value=50, min=0, max=10000, step=0.1, description='R (Ohms)')
x_s = widgets.FloatSlider(value=25, min=-5000, max=10000, step=0.1, description='X (Ohms)')
z_c = widgets.Checkbox(value=True, description='Z Grid (Red)')
y_c = widgets.Checkbox(value=True, description='Y Grid (Blue)')

ui = widgets.VBox([widgets.HBox([r_s, x_s]), widgets.HBox([z_c, y_c])])
out = widgets.interactive_output(showSmithBasics, {'r_val': r_s, 'x_val': x_s, 'show_z': z_c, 'show_y': y_c})

display(ui, out)

Output()

## Parallel and Series 

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import skrf as rf
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# --- ESTADO GLOBAL ---
state = {
    'z_current_norm': complex(1.0, 0.0),
    'history': [] # Lista de diccionarios con {path, label}
}

def calculate_path(z_start_norm, type_sel, val):
    """Calcula los puntos del arco para la transformación"""
    if 'Series' in type_sel:
        x_val = val if '+x' in type_sel else -val
        z_end_norm = z_start_norm + complex(0, x_val)
        p_type = 'z'
    else:
        b_val = val if '+b' in type_sel else -val
        y_s_norm = 1 / z_start_norm if z_start_norm != 0 else complex(1e6, 0)
        y_e_norm = y_s_norm + complex(0, b_val)
        z_end_norm = 1 / y_e_norm
        p_type = 'y'
        
    pts = 50
    path = []
    for i in range(pts + 1):
        t = i / pts
        if p_type == 'z':
            z_int = complex(z_start_norm.real, z_start_norm.imag + t * (z_end_norm.imag - z_start_norm.imag))
        else:
            y_s, y_e = 1/z_start_norm, 1/z_end_norm
            z_int = 1 / complex(y_s.real, y_s.imag + t * (y_e.imag - y_s.imag))
        path.append((z_int - 1) / (z_int + 1))
    return np.array(path), z_end_norm

def redraw(change=None):
    with out_plot:
        clear_output(wait=True)
        plt.close('all')
        fig, ax = plt.subplots(figsize=(8, 8))
        
        # 1. Grillas Base (Red/Blue)
        rf.plotting.smith(ax=ax, chart_type='z', draw_labels=True)
        for l in ax.lines: l.set_color('red'); l.set_alpha(0.15)
        n_z = len(ax.lines)
        rf.plotting.smith(ax=ax, chart_type='y', draw_labels=False)
        for l in ax.lines[n_z:]: l.set_color('blue'); l.set_alpha(0.1)

        # 2. Dibujar Historial (Arcos anteriores)
        for idx, item in enumerate(state['history']):
            ax.plot(item['path'].real, item['path'].imag, color='black', lw=1.5, alpha=0.5)
            # Pequeña marca al final de cada paso previo
            ax.plot(item['path'][-1].real, item['path'][-1].imag, 'ko', markersize=4)

        # 3. Dibujar Preview (Paso actual)
        path_preview, z_end_preview = calculate_path(state['z_current_norm'], type_w.value, val_w.value)
        ax.plot(path_preview.real, path_preview.imag, color='darkgreen', lw=3, zorder=10)
        ax.plot(path_preview[0].real, path_preview[0].imag, 'ro', markersize=8, label='Punto Actual')
        ax.plot(path_preview[-1].real, path_preview[-1].imag, 'go', markersize=10, label='Preview')
        
        # Flecha de dirección en el preview
        mid = 30
        ax.annotate('', xy=(path_preview[mid].real, path_preview[mid].imag), 
                    xytext=(path_preview[mid-1].real, path_preview[mid-1].imag),
                    arrowprops=dict(arrowstyle='->', lw=2, color='darkgreen'))

        ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_aspect('equal')
        ax.legend(loc='upper right', fontsize=9)
        plt.show()
        
        # 4. Mostrar Bitácora
        if state['history']:
            print("--- BITÁCORA DE DISEÑO ---")
            for i, h in enumerate(state['history']):
                print(f"{i+1}. {h['label']}")
        
        y_fin = 1/z_end_preview
        display(Markdown(fr"**Z final actual:** {z_end_preview.real*50:.1f} + j{z_end_preview.imag*50:.1f} $\Omega$ | "
                         fr"**y:** {y_fin.real:.2f} + j{y_fin.imag:.2f}"))

def on_add_click(b):
    path, z_end = calculate_path(state['z_current_norm'], type_w.value, val_w.value)
    # Guardar en historial
    label = f"{type_w.value} (val={val_w.value:.2f})"
    state['history'].append({'path': path, 'label': label})
    state['z_current_norm'] = z_end
    val_w.value = 0 # Reset para el próximo elemento
    redraw()

def on_reset_click(b):
    state['z_current_norm'] = complex(r_init_w.value/50, x_init_w.value/50)
    state['history'] = []
    redraw()

# --- INTERFAZ ---
out_plot = widgets.Output()
r_init_w = widgets.FloatSlider(value=50, min=0.1, max=500, description='R load (Ω)')
x_init_w = widgets.FloatSlider(value=0, min=-500, max=500, description='X load (Ω)')
type_w = widgets.Dropdown(options=['Series L (+x)', 'Series C (-x)', 'Shunt C (+b)', 'Shunt L (-b)'], description='Componente:')
val_w = widgets.FloatSlider(value=0.5, min=0, max=5, step=0.01, description='Valor (x/b)')
btn_add = widgets.Button(description="Agregar Componente", button_style='success', icon='plus')
btn_reset = widgets.Button(description="Reset a Carga Inicial", button_style='danger', icon='refresh')

btn_add.on_click(on_add_click)
btn_reset.on_click(on_reset_click)
for w in [type_w, val_w]: w.observe(redraw, names='value')

ui = widgets.VBox([
    widgets.HTML("<b>1. Definir Carga de inicio y pulsar Reset:</b>"),
    widgets.HBox([r_init_w, x_init_w, btn_reset]),
    widgets.HTML("<b>2. Ajustar preview y Agregar para encadenar:</b>"),
    widgets.HBox([type_w, val_w, btn_add]),
    out_plot
])

display(ui)
on_reset_click(None)

## L Network 

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import skrf as rf
import ipywidgets as widgets
from IPython.display import display, Markdown

def updateSmithUI(rsS, xsS, rlL, xlL, freqMhz=100, z0=50, filter_type='LPF'):
    # 1. Parámetros de diseño
    w = 2 * np.pi * freqMhz * 1e6
    zLoad = complex(rlL, xlL)
    zTarget = np.conj(complex(rsS, xsS))
    
    # 2. Cálculo de Intersección
    yLoad = 1 / zLoad
    gL, bL = yLoad.real, yLoad.imag
    gTarget = 1 / rsS
    
    # Decidir dirección del arco según el tipo de filtro
    # LPF: bMatch > 0 (Cap), xMatch > 0 (Ind)
    # HPF: bMatch < 0 (Ind), xMatch < 0 (Cap)
    sign = 1 if filter_type == 'LPF' else -1
    
    # Cálculo de la susceptancia de la red shunt
    bMatch = sign * np.sqrt(max(0, gL * (gTarget - gL))) - bL
    yInter = complex(gL, bL + bMatch)
    zInter = 1 / yInter
    
    # Cálculo de la reactancia de la red serie
    xMatch = zTarget.imag - zInter.imag
    
    # Identificación de componentes
    if filter_type == 'LPF':
        comp1_name, comp1_val = "Capacitor Shunt (C)", (bMatch / w) * 1e12
        comp1_unit = "pF"
        comp2_name, comp2_val = "Inductor Serie (L)", (xMatch / w) * 1e9
        comp2_unit = "nH"
    else:
        comp1_name, comp1_val = "Inductor Shunt (L)", (-1 / (bMatch * w)) * 1e9
        comp1_unit = "nH"
        comp2_name, comp2_val = "Capacitor Serie (C)", (-1 / (xMatch * w)) * 1e12
        comp2_unit = "pF"

    # 3. Generación de Arcos
    def getArcData(zStart, zEnd, stepType='z', numPoints=100):
        gammaPoints = []
        if stepType == 'y': 
            yStart, yEnd = 1/zStart, 1/zEnd
            bValues = np.linspace(yStart.imag, yEnd.imag, numPoints)
            for b in bValues:
                zPoint = 1 / complex(yStart.real, b)
                gammaPoints.append((zPoint - z0) / (zPoint + z0))
        else: 
            xValues = np.linspace(zStart.imag, zEnd.imag, numPoints)
            for x in xValues:
                zPoint = complex(zStart.real, x)
                gammaPoints.append((zPoint - z0) / (zPoint + z0))
        pts = np.array(gammaPoints)
        arcLength = np.sum(np.abs(np.diff(pts)))
        return pts, arcLength

    arcShunt, lenShunt = getArcData(zLoad, zInter, stepType='y')
    arcSerie, lenSerie = getArcData(zInter, zTarget, stepType='z')

    # 4. CONFIGURACIÓN DEL GRÁFICO (10x10)
    fig, ax = plt.subplots(figsize=(10, 10)) 
    plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
    
    rf.plotting.smith(ax=ax, chart_type='z', draw_labels=True)
    
    zLines = ax.get_lines()
    for line in zLines:
        line.set_color('royalblue'); line.set_alpha(0.3); line.set_linewidth(1.0)
    
    numZ = len(zLines)
    rf.plotting.smith(ax=ax, chart_type='y', draw_labels=False)
    allL = ax.get_lines()
    for i in range(numZ, len(allL)):
        allL[i].set_color('crimson'); allL[i].set_linestyle('--'); allL[i].set_alpha(0.2); allL[i].set_linewidth(1.0)

    # 5. Dibujar Arcos
    ax.plot(arcShunt.real, arcShunt.imag, color='black', linewidth=3, label=f'1. {comp1_name}', zorder=10)
    midS = len(arcShunt) // 2
    ax.annotate('', xy=(arcShunt[midS+1].real, arcShunt[midS+1].imag), 
                xytext=(arcShunt[midS].real, arcShunt[midS].imag),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'), zorder=12)

    ax.plot(arcSerie.real, arcSerie.imag, color='darkgreen', linewidth=3, label=f'2. {comp2_name}', zorder=10)
    midE = len(arcSerie) // 2
    ax.annotate('', xy=(arcSerie[midE+1].real, arcSerie[midE+1].imag), 
                xytext=(arcSerie[midE].real, arcSerie[midE].imag),
                arrowprops=dict(arrowstyle='->', lw=2, color='darkgreen'), zorder=12)

    gL = (zLoad - z0) / (zLoad + z0)
    gT = (zTarget - z0) / (zTarget + z0)
    ax.plot([gL.real], [gL.imag], 'ro', markersize=10, label='ZL', markeredgecolor='black', zorder=15)
    ax.plot([gT.real], [gT.imag], 'gx', markersize=12, mew=3, label='Zs*', zorder=15)

    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect('equal')
    
    for text in ax.texts:
        text.set_fontsize(9)

    ax.set_title(f"Diseño Red L ({filter_type}) @ {freqMhz} MHz", fontsize=14, pad=20)
    ax.legend(loc='upper right', fontsize=10, markerscale=0.8, handlelength=1.0, frameon=True, borderpad=0.3)
    plt.show()

    # 6. Tabla de Resultados
    display(Markdown("---"))
    display(Markdown(f"## 🛠️ Parámetros de la Red de Acoplamiento ({filter_type})"))
    display(Markdown(fr"| Componente | Valor Calculado | Longitud del Arco ($\Delta\Gamma$) |"))
    display(Markdown(fr"| :--- | :--- | :--- |"))
    display(Markdown(fr"| **{comp1_name}** | `{comp1_val:.2f} {comp1_unit}` | `{lenShunt:.4f}` |"))
    display(Markdown(fr"| **{comp2_name}** | `{comp2_val:.2f} {comp2_unit}` | `{lenSerie:.4f}` |"))

# Sliders y Dropdown
rsSlider = widgets.FloatSlider(value=50, min=10, max=300, description='Rs (Ω)')
xsSlider = widgets.FloatSlider(value=0, min=-100, max=100, description='Xs (Ω)')
rlSlider = widgets.FloatSlider(value=150, min=10, max=1000, description='Rl (Ω)')
xlSlider = widgets.FloatSlider(value=0, min=-500, max=500, description='Xl (Ω)')
freqSlider = widgets.FloatSlider(value=100, min=1, max=2400, description='Freq (MHz)')
filterType = widgets.Dropdown(options=['LPF', 'HPF'], value='LPF', description='Tipo Filtro')

ui = widgets.VBox([
    widgets.HBox([rsSlider, xsSlider]), 
    widgets.HBox([rlSlider, xlSlider]), 
    widgets.HBox([freqSlider, filterType])
])

out = widgets.interactive_output(updateSmithUI, {
    'rsS': rsSlider, 'xsS': xsSlider, 'rlL': rlSlider, 'xlL': xlSlider, 
    'freqMhz': freqSlider, 'filter_type': filterType
})

display(ui, out)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1000x1000 with 1 Axes>', …